# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [01:46<00:00, 35.42s/it]


In [4]:
len(deals)

30

In [5]:
deals[10].describe()

"Title: Lexar 512GB Play PRO microSD Express Card for $101 + free shipping\nDetails: That's the best price we found by $9. Buy Now at Amazon\nFeatures: read speeds up to 900MB/s write speed up to 600MB/s backwards-compatible with UHS-I and UHS-II (at UHS-I speeds) Model: LMSXPS0512G-BNNNU\nURL: https://www.dealnews.com/products/Lexar/Lexar-512-GB-Play-PRO-micro-SD-Express-Card/497910.html?iref=rss-c39"

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [6]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [7]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [8]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Anker Prime 250W Prime Charging Station for $100 + free shipping
Details: That ties the lowest price we've seen at $50 off. Buy Now at Amazon
Features: 2.26" LCD display four USB-C and two USB-A ports Model: A2345
URL: https://www.dealnews.com/products/Anker/Anker-Prime-250-W-Prime-Charging-Station/479399.html?iref=rss-c142

Title: Samsung 43" Class Q7F QLED TV: Free w/ Fios Ho

In [9]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='Anker Solix C1000 is a high-capacity portable power station designed for RV use and outdoor camping. It features an 1,800W continuous (2,400W peak) output delivered through 11 ports, supports up to 600W solar input for fast recharging, and uses an LFP battery rated for about 3,000 cycles. The unit supports rapid charging — to 80% in ~43 minutes and full charge under an hour — and includes app monitoring via Bluetooth and Wi‑Fi as well as a 5-year warranty.', price=429.0, url='https://www.dealnews.com/Anker-Solix-C1000-1-800-W-Portable-Power-Station-for-429-free-shipping/21815138.html?iref=rss-c142'), Deal(product_description='MOVA Z60 Ultra Roller Complete is a high-end robot vacuum with extremely strong 28,000Pa suction and an integrated roller-mop system that provides always-clean roller washing for streak-free floors. It includes AutoShield carpet protection to block moisture, obstacle-climbing capability up to 8 cm, and expansive cover

In [10]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


Anker Solix C1000 is a high-capacity portable power station designed for RV use and outdoor camping. It features an 1,800W continuous (2,400W peak) output delivered through 11 ports, supports up to 600W solar input for fast recharging, and uses an LFP battery rated for about 3,000 cycles. The unit supports rapid charging — to 80% in ~43 minutes and full charge under an hour — and includes app monitoring via Bluetooth and Wi‑Fi as well as a 5-year warranty.
429.0
https://www.dealnews.com/Anker-Solix-C1000-1-800-W-Portable-Power-Station-for-429-free-shipping/21815138.html?iref=rss-c142

MOVA Z60 Ultra Roller Complete is a high-end robot vacuum with extremely strong 28,000Pa suction and an integrated roller-mop system that provides always-clean roller washing for streak-free floors. It includes AutoShield carpet protection to block moisture, obstacle-climbing capability up to 8 cm, and expansive coverage aided by extendable roller and side brush reaching wide areas for comprehensive clean

In [11]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [12]:
from agents.scanner_agent import ScannerAgent

In [13]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [14]:
result

DealSelection(deals=[Deal(product_description='Anker Solix C1000 is a portable power station designed for heavy-duty outdoor and RV use. It delivers 1,800W (2,400W peak) AC output across 11 ports, supports up to 600W solar input for fast recharging, and charges to 80% in 43 minutes (full charge under one hour). The unit uses an LFP battery rated for 3,000 cycles, includes app monitoring via Bluetooth and Wi‑Fi, and comes with a 5‑year warranty for long-term reliability.', price=429.0, url='https://www.dealnews.com/Anker-Solix-C1000-1-800-W-Portable-Power-Station-for-429-free-shipping/21815138.html?iref=rss-c142'), Deal(product_description='Refurbished Dell OptiPlex 7090 UFF is an ultra‑small form desktop equipped with an 11th‑Generation Intel Core i5‑1145G7 quad‑core CPU, 32GB of RAM, and a 256GB SSD. The system ships with Windows 10 Professional and includes a 100‑day Dell warranty, making it suitable for compact office setups or home labs where space is limited but multitasking perfo

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [ ]:
load_dotenv(override=True)

In [ ]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [ ]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

In [ ]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
push("MASSIVE DEAL!!")

In [ ]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

In [ ]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")